# Bài 6.3 · BERT tiếng Việt trên dữ liệu thật

**Câu hỏi của buổi này:** một model ngôn ngữ đã học sẵn tiếng Việt (PhoBERT) hơn cách đếm từ (TF-IDF) bao nhiêu, và hơn ở đâu?

**UIT-VSFC** gồm hơn 16 000 câu phản hồi của sinh viên về môn học và giảng viên, gán nhãn tiêu cực, trung tính, tích cực. Nhãn trung tính chỉ khoảng 4%, nên metric là **macro-F1**: mỗi lớp góp phần ngang nhau.

| Mục | Sẽ thấy |
|---|---|
| 1. Dữ liệu | bảng ba tập, tỉ lệ ba nhãn, câu mẫu |
| 2. Baseline TF-IDF | macro-F1, những từ model dựa vào nhiều nhất |
| 3. Tách từ và tokenizer | một câu qua từng bước, phân bố độ dài |
| 4. Fine-tune PhoBERT | loss theo bước, macro-F1 validation từng epoch |
| 5. Chấm test | bảng và biểu đồ F1 từng lớp, hai confusion matrix |
| 6. Câu bị đoán sai và câu tự viết | bảng câu thật, xác suất ba lớp |
| 7. Tóm tắt | một bảng các con số chính |

Các dòng ✓ là phép kiểm in ra: chạy lại mà thấy ✗ thì có gì đó sai. Bật **GPU** và **Internet** trong *Session options*; chạy hết khoảng 8 phút. Bài giảng: [ai-canuc80k · bài 6.3](https://ai-canuc80k.ai-web-deploy-20261010.workers.dev/#/lesson/6.3).

In [ ]:
!pip install -q pyvi > /dev/null 2>&1

In [ ]:
# Ô tiện ích: định dạng bảng, dòng kiểm ✓/✗, nhãn số trên biểu đồ. Không cần đọc để theo bài.
import warnings, contextlib, io, logging
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display, HTML

plt.rcParams.update({
    'figure.dpi': 110, 'figure.figsize': (8, 4), 'font.size': 11, 'axes.titlesize': 12, 'axes.titleweight': 'bold',
    'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': True, 'grid.alpha': .25,
    'legend.frameon': False, 'image.cmap': 'viridis',
})
C = {'blue': '#3b6ea8', 'clay': '#c2603e', 'sand': '#b8892b', 'green': '#4d7337', 'gray': '#8a877f'}
ORDER = [C['blue'], C['clay'], C['sand'], C['green'], C['gray']]
plt.rcParams['axes.prop_cycle'] = plt.cycler(color=ORDER)


def table(df, title=None, fmt=None, best=None, lower_is_better=False, index=False):
    """In một bảng gọn: định dạng số, tô đậm ô tốt nhất của cột `best`."""
    st = df.style.format(fmt, na_rep='–')
    if best is not None:
        col = df[best]
        target = col.min() if lower_is_better else col.max()
        st = st.apply(lambda s: ['font-weight:700;background:#f5e6dd' if (s.name == best and v == target) else '' for v in s], axis=0)
    text_cols = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])]
    if text_cols:
        st = st.set_properties(subset=text_cols, **{'text-align': 'left'})
    if not index:
        st = st.hide(axis='index')
    st = st.set_table_styles([
        {'selector': 'th', 'props': 'text-align:left;padding:6px 12px;border-bottom:2px solid #d3cebf;font-weight:600'},
        {'selector': 'td', 'props': 'padding:5px 12px;border-bottom:1px solid #e6e2d6'},
        {'selector': 'caption', 'props': 'caption-side:top;text-align:left;font-weight:700;font-size:14px;padding:4px 0 8px;white-space:nowrap'},
    ])
    if title:
        st = st.set_caption(title)
    display(st)


def check(msg, ok):
    """Dòng kiểm tra in ra: ✓ khi đúng, ✗ khi sai (không dừng notebook)."""
    print(('✓ ' if ok else '✗ ') + msg)


def bar_labels(ax, fmt='{:.2f}'):
    """Ghi giá trị lên đầu mỗi thanh của biểu đồ cột."""
    for cont in ax.containers:
        ax.bar_label(cont, labels=[fmt.format(v) for v in cont.datavalues], padding=3, fontsize=9)


@contextlib.contextmanager
def quiet():
    """Ẩn thanh tiến trình và dòng thông báo khi tải dữ liệu hay trọng số."""
    logging.disable(logging.WARNING)
    try:
        with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
            yield
    finally:
        logging.disable(logging.NOTSET)


def note(text):
    """Một dòng nhận xét in đậm nhẹ dưới output, tính từ số thật của lần chạy."""
    display(HTML(f'<div style="border-left:3px solid #c2603e;padding:4px 10px;margin:6px 0;color:#444">{text}</div>'))

In [ ]:
import time, copy
import torch, transformers
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix
from pyvi import ViTokenizer
transformers.logging.set_verbosity_error()

torch.manual_seed(0); np.random.seed(0)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
LABELS = ['tiêu cực', 'trung tính', 'tích cực']
table(pd.DataFrame([('torch', torch.__version__), ('transformers', transformers.__version__),
                    ('thiết bị', torch.cuda.get_device_name(0) if device == 'cuda' else 'CPU')],
                   columns=['thành phần', 'phiên bản']), 'Môi trường chạy')

## 1. Dữ liệu

Ba tập train, validation, test đã chia sẵn, tải dạng parquet từ Hugging Face. Cột `sentiment`: 0 tiêu cực, 1 trung tính, 2 tích cực.

In [ ]:
URL = 'https://huggingface.co/api/datasets/uitnlp/vietnamese_students_feedback/parquet/default/{}/0.parquet'
df = {s: pd.read_parquet(URL.format(s)) for s in ['train', 'validation', 'test']}
dist = pd.DataFrame({s: d['sentiment'].value_counts(normalize=True).sort_index().values for s, d in df.items()}, index=LABELS).T
dist.insert(0, 'số câu', [len(d) for d in df.values()])
table(dist.reset_index(names='tập'), 'Ba tập và tỉ lệ ba nhãn', fmt={l: '{:.1%}' for l in LABELS})
check('lớp trung tính dưới 6% ở cả ba tập', (dist['trung tính'] < 0.06).all())
table(pd.DataFrame([(LABELS[k], df['train'][df['train'].sentiment == k].sentence.iloc[j]) for k in range(3) for j in range(2)],
                   columns=['nhãn', 'câu']), 'Câu mẫu trong train')

## 2. Baseline: TF-IDF và hồi quy logistic

Đếm từ và cặp từ, đánh trọng số TF-IDF, rồi phân loại tuyến tính. `class_weight='balanced'` tăng trọng số lớp trung tính hiếm. Đây là mốc PhoBERT phải vượt.

In [ ]:
tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
Xtr = tfidf.fit_transform(df['train'].sentence); Xte = tfidf.transform(df['test'].sentence)
lr = LogisticRegression(max_iter=3000, class_weight='balanced').fit(Xtr, df['train'].sentiment)
pred_tfidf = lr.predict(Xte)
f1_tfidf = f1_score(df['test'].sentiment, pred_tfidf, average='macro')

vocab = np.array(tfidf.get_feature_names_out())
fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
for k, ax in enumerate(axes):
    top = np.argsort(lr.coef_[k])[-8:]
    ax.barh(vocab[top], lr.coef_[k][top], color=[C['clay'], C['gray'], C['green']][k]); ax.grid(axis='y', visible=False)
    ax.set_title(f'từ đẩy về "{LABELS[k]}"', fontsize=11)
plt.tight_layout(); plt.show()
note(f'TF-IDF + logistic: macro-F1 test <b>{f1_tfidf:.4f}</b>. Model chỉ nhìn từ nào xuất hiện, không biết thứ tự hay phủ định ("không hay" vẫn có từ "hay").')

## 3. Tách từ tiếng Việt và tokenizer của PhoBERT

Tiếng Việt viết cách theo âm tiết: "giảng viên" là một từ hai âm tiết. PhoBERT học trên văn bản đã tách từ, các âm tiết của một từ nối bằng dấu gạch dưới. `pyvi` làm bước này, rồi tokenizer BPE của PhoBERT cắt tiếp thành subword.

In [ ]:
with quiet():
    tokenizer = AutoTokenizer.from_pretrained('vinai/phobert-base')
s = 'giảng viên nhiệt tình , bài giảng dễ hiểu nhưng phòng học hơi nóng .'
seg1 = ViTokenizer.tokenize(s)
table(pd.DataFrame([('câu gốc', s), ('tách từ (pyvi)', seg1), ('token PhoBERT', ' | '.join(tokenizer.tokenize(seg1))),
                    ('id', ' '.join(map(str, tokenizer(seg1)['input_ids'])))], columns=['bước', 'kết quả']), 'Một câu qua từng bước')

t = time.time()
seg = {k: [ViTokenizer.tokenize(x) for x in d.sentence] for k, d in df.items()}
lens = np.array([len(tokenizer(x)['input_ids']) for x in seg['train']])
MAX_LEN = 96
fig, ax = plt.subplots(figsize=(9, 3.2))
ax.hist(lens, bins=60, color=C['blue']); ax.axvline(MAX_LEN, color=C['clay'], ls='--'); ax.text(MAX_LEN + 1, ax.get_ylim()[1] * .8, f'cắt ở {MAX_LEN}', color=C['clay'])
ax.set(xlabel='số token mỗi câu (kể cả <s> và </s>)', ylabel='số câu', title=f'Độ dài câu train (tách từ mất {time.time() - t:.0f} giây)')
plt.tight_layout(); plt.show()
check(f'chỉ {np.mean(lens > MAX_LEN):.2%} câu dài hơn {MAX_LEN} token', np.mean(lens > MAX_LEN) < 0.01)
note(f'Trung vị <b>{int(np.median(lens))}</b> token, phân vị 99% là {int(np.percentile(lens, 99))}: câu phản hồi ngắn, cắt ở {MAX_LEN} gần như không mất gì.')

## 4. Fine-tune PhoBERT

Mỗi batch chỉ đệm tới câu dài nhất của batch đó (`tokenizer.pad`). Learning rate tăng dần trong 10% bước đầu rồi giảm tuyến tính. Sau mỗi epoch chấm macro-F1 trên validation và giữ bản tốt nhất; test chưa được đụng tới.

In [ ]:
def make_loader(split, shuffle, bs=32):
    enc = tokenizer(seg[split], truncation=True, max_length=MAX_LEN)
    items = [{'input_ids': i, 'attention_mask': a, 'labels': int(y)} for i, a, y in zip(enc['input_ids'], enc['attention_mask'], df[split].sentiment)]
    return DataLoader(items, batch_size=bs, shuffle=shuffle, collate_fn=lambda b: tokenizer.pad(b, return_tensors='pt'))

loaders = {s: make_loader(s, s == 'train') for s in df}
with quiet():
    model = AutoModelForSequenceClassification.from_pretrained('vinai/phobert-base', num_labels=3).to(device)
EPOCHS = 3
opt = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
steps = EPOCHS * len(loaders['train'])
sched = get_linear_schedule_with_warmup(opt, int(0.1 * steps), steps)
scaler = torch.amp.GradScaler('cuda', enabled=device == 'cuda')

def predict(loader, proba=False):
    model.eval(); out = []
    with torch.no_grad(), torch.autocast(device, dtype=torch.float16, enabled=device == 'cuda'):
        for b in loader:
            b = {k: v.to(device) for k, v in b.items() if k != 'labels'}
            lg = model(**b).logits.float()
            out.append(lg.softmax(-1).cpu() if proba else lg.argmax(-1).cpu())
    return torch.cat(out).numpy()

best_f1, best_state, step_loss, ep_rows = -1, None, [], []
for ep in range(EPOCHS):
    t = time.time(); model.train()
    for b in loaders['train']:
        b = {k: v.to(device) for k, v in b.items()}
        with torch.autocast(device, dtype=torch.float16, enabled=device == 'cuda'):
            loss = model(**b).loss
        opt.zero_grad(set_to_none=True); scaler.scale(loss).backward()
        scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt); scaler.update(); sched.step()
        step_loss.append(loss.item())
    f1 = f1_score(df['validation'].sentiment, predict(loaders['validation']), average='macro')
    ep_rows.append((ep + 1, np.mean(step_loss[-len(loaders['train']):]), f1, time.time() - t))
    if f1 > best_f1:
        best_f1, best_ep, best_state = f1, ep + 1, copy.deepcopy({k: v.cpu() for k, v in model.state_dict().items()})
model.load_state_dict(best_state)
E = pd.DataFrame(ep_rows, columns=['epoch', 'loss train trung bình', 'macro-F1 validation', 'giây'])
table(E, f'Fine-tune {EPOCHS} epoch, giữ epoch {best_ep}', fmt={'loss train trung bình': '{:.3f}', 'macro-F1 validation': '{:.4f}', 'giây': '{:.0f}'}, best='macro-F1 validation')

fig, ax = plt.subplots(figsize=(10, 3.4))
ax.plot(step_loss, color=C['gray'], lw=.5, alpha=.5, label='từng bước')
ax.plot(pd.Series(step_loss).rolling(50, min_periods=1).mean(), color=C['blue'], lw=2, label='trung bình trượt 50 bước')
for k in range(1, EPOCHS): ax.axvline(k * len(loaders['train']), color='#999', ls=':', lw=1)
ax.set(xlabel='bước cập nhật (đường chấm: hết một epoch)', ylabel='loss', title='Loss train của PhoBERT'); ax.legend()
plt.tight_layout(); plt.show()

## 5. Chấm test một lần

In [ ]:
y_te = df['test'].sentiment.to_numpy()
pred = predict(loaders['test'])
f1_bert = f1_score(y_te, pred, average='macro')
res = pd.DataFrame([(n, accuracy_score(y_te, p), f1_score(y_te, p, average='macro'), *f1_score(y_te, p, average=None)) for n, p in [('TF-IDF + logistic', pred_tfidf), ('PhoBERT', pred)]],
                   columns=['model', 'accuracy', 'macro-F1'] + [f'F1 {l}' for l in LABELS])
table(res, f'Test ({len(y_te)} câu)', fmt={c: '{:.3f}' for c in res.columns[1:]}, best='macro-F1')

fig, axes = plt.subplots(1, 3, figsize=(16, 4.4), gridspec_kw={'width_ratios': [1.2, 1, 1]})
x = np.arange(3)
for k, (n, c) in enumerate(zip(res['model'], [C['gray'], C['blue']])):
    axes[0].bar(x + (k - .5) * .38, res.iloc[k, 3:].astype(float), .38, color=c, label=n)
axes[0].set_xticks(x, LABELS); axes[0].set_ylim(0, 1.25); axes[0].set_title('F1 từng lớp'); axes[0].legend(loc='upper center', ncol=2); bar_labels(axes[0], '{:.2f}')
for ax, (n, p) in zip(axes[1:], [('TF-IDF + logistic', pred_tfidf), ('PhoBERT', pred)]):
    cm = confusion_matrix(y_te, p); cmn = cm / cm.sum(1, keepdims=True)
    ax.imshow(cmn, cmap='Blues', vmin=0, vmax=1); ax.grid(False)
    for i in range(3):
        for j in range(3):
            ax.text(j, i, f'{cm[i, j]}\n{cmn[i, j]:.0%}', ha='center', va='center', fontsize=10, color='white' if cmn[i, j] > .5 else '#222')
    ax.set_xticks(range(3), LABELS); ax.set_yticks(range(3), LABELS); ax.set(xlabel='dự đoán', ylabel='nhãn thật', title=n)
plt.tight_layout(); plt.show()
check('PhoBERT vượt TF-IDF về macro-F1', f1_bert > f1_tfidf)
neu = res['F1 trung tính']
note(f"macro-F1: TF-IDF {f1_tfidf:.3f} → PhoBERT <b>{f1_bert:.3f}</b>; phần lớn nhờ lớp trung tính, F1 từ {neu.iloc[0]:.2f} lên {neu.iloc[1]:.2f}, dù vẫn là lớp khó nhất. "
     f"Chỉ nhìn accuracy ({res['accuracy'].iloc[1]:.3f}) thì không thấy chỗ yếu này, macro-F1 thì thấy.")

## 6. Câu bị đoán sai và câu tự viết

In [ ]:
proba = predict(loaders['test'], proba=True)
wrong = np.flatnonzero(pred != y_te)
pick = wrong[np.argsort(-proba[wrong].max(1))][:8]
table(pd.DataFrame({'câu': df['test'].sentence.iloc[pick].values, 'nhãn thật': [LABELS[i] for i in y_te[pick]],
                    'PhoBERT đoán': [LABELS[i] for i in pred[pick]], 'độ tin': proba[pick].max(1)}),
      f'Tám câu sai mà PhoBERT tự tin nhất (trong {len(wrong)} câu sai)', fmt={'độ tin': '{:.2f}'})

mine = ['thầy giảng rất dễ hiểu và nhiệt tình', 'phòng học nóng quá , máy chiếu hay bị hỏng', 'môn này có 3 tín chỉ',
        'bài tập nhiều nhưng học được nhiều thứ', 'không hay lắm']
enc = tokenizer([ViTokenizer.tokenize(x) for x in mine], padding=True, return_tensors='pt').to(device)
model.eval()
with torch.no_grad():
    pm = model(**enc).logits.float().softmax(-1).cpu().numpy()
fig, ax = plt.subplots(figsize=(10, 3))
left = np.zeros(len(mine))
for k, c in enumerate([C['clay'], C['gray'], C['green']]):
    ax.barh(mine, pm[:, k], left=left, color=c, label=LABELS[k]); left += pm[:, k]
ax.invert_yaxis(); ax.set_xlim(0, 1); ax.set_title('Câu tự viết: xác suất ba lớp'); ax.legend(loc='upper left', bbox_to_anchor=(1, 1)); ax.grid(axis='y', visible=False)
plt.tight_layout(); plt.show()

## 7. Tóm tắt

In [ ]:
summary = [('tỉ lệ câu trung tính (train)', f"{dist['trung tính'].iloc[0]:.1%}"),
           ('macro-F1 test: TF-IDF / PhoBERT', f'{f1_tfidf:.3f} / {f1_bert:.3f}'),
           ('F1 lớp trung tính: TF-IDF / PhoBERT', f'{neu.iloc[0]:.3f} / {neu.iloc[1]:.3f}'),
           ('accuracy PhoBERT', f"{res['accuracy'].iloc[1]:.3f}"),
           ('epoch được giữ (theo validation)', f'{best_ep} / {EPOCHS}')]
table(pd.DataFrame(summary, columns=['điều đo', 'kết quả']), 'Các con số chính của notebook')

**Thử thêm khi dạy:** thêm trọng số lớp vào loss (`torch.nn.CrossEntropyLoss(weight=...)`) để nâng lớp trung tính; bỏ bước tách từ pyvi rồi fine-tune lại xem macro-F1 giảm bao nhiêu; thêm câu khen chê lẫn lộn vào danh sách `mine`.